In [0]:
import requests
import pandas as pd
import time

from bs4 import BeautifulSoup

all_data = []

start = 0
count = 100

while True:

    url = (
        f"https://finance.yahoo.com/markets/stocks/most-active/"
        f"?start={start}&count={count}"
    )

    print(f"Scraping page starting at {start}")

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers
    )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    rows = soup.select("table tbody tr")

    if len(rows) == 0:
        break

    for row in rows:

        cols = row.find_all("td")

        if len(cols) < 8:
            continue

        all_data.append({
            "ticker": cols[0].text.strip(),
            "company_name": cols[1].text.strip(),
            "price": cols[2].text.strip(),
            "change": cols[3].text.strip(),
            "change_percent": cols[4].text.strip(),
            "volume": cols[5].text.strip(),
            "avg_volume_3m": cols[6].text.strip(),
            "market_cap": cols[7].text.strip(),
            "PE_ratio": cols[8].text.strip(),
            "52_wk_change%": cols[9].text.strip(),
            "52_Wk_Range": cols[10].text.strip(),
        })

    start += count

    time.sleep(2)

market_df = pd.DataFrame(all_data)

print(market_df.head())


from pyspark.sql.functions import (
    current_timestamp
)

spark_df = spark.createDataFrame(
    market_df
)

spark_df = spark_df.withColumn(
    "dwh_loaded_at",
    current_timestamp()
)
spark.sql("""DROP table if exists finance_intelligence_hub.bronze.market_activity_raw""")

spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "finance_intelligence_hub.bronze.market_activity_raw"
    )

In [0]:
%sql
Select * from finance_intelligence_hub.bronze.market_activity_raw;